In [1]:
# from huggingface_hub import hf_hub_download

# hf_hub_download(
#     repo_id="mesolitica/Malaysian-TTS-v2", 
#     filename="neucodec.zip",
#     local_dir="./Malaysian-TTS-v2",
#     repo_type="dataset",
# )

# hf_hub_download(
#     repo_id="mesolitica/Malaysian-TTS-v2", 
#     filename="processed.parquet",
#     local_dir="./Malaysian-TTS-v2",
#     repo_type="dataset",
# )

In [18]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
# !unzip neucodec.zip

In [6]:
import pandas as pd

df = pd.read_parquet('processed.parquet')
df.head()

,reference_text,generate_text,normalized_generate_text,reference_audio,filename_audio,speaker,similarity,audio_length,index,alignment,averaged_pitch,distances
0,"Uhm, hello, selamat pagi ye, saya dari custome...","Encik, bolehkah Encik memberikan maklum balas ...","Encik, bolehkah Encik memberikan maklum balas ...",husein-assistant.mp3,response-husein-v3/55420.mp3,husein,0.800952,4.400181,1367694,"[{'end': 0.38, 'score': -3.89, 'start': 0.12, ...","[279.365, 104.309, 97.318, 94.798, 95.233, 91....","[0.073, 0.007, 0.012, 0.006, 0.007, 0.012, 0.0..."
1,"Hi, saya adalah pembantu AI anda, selamat berk...","Puan, saya dari DanceFit, uhm, kami ada kelas ...","Puan, saya dari DanceFit, uhm, kami ada kelas ...",shafiqah-idayu-enhanced-v2-v2-trim.mp3,introduction-idayu-v2/102406.mp3,idayu,0.737765,7.291066,1772554,"[{'end': 0.72, 'score': -1.533, 'start': 0.28,...","[248.865, 237.913, 218.932, 253.237, 209.599, ...","[0.016, 0.06, 0.02, 0.004, 0.065, 0.035, 0.027..."
2,"Uhm, hello, selamat pagi ye, saya dari custome...","Cik, rider sedang dalam perjalanan untuk hanta...","Cik, rider sedang dalam perjalanan untuk hanta...",husein-assistant.mp3,response-husein-v2/107749.mp3,husein,0.731988,6.153288,1218381,"[{'end': 0.36, 'score': -1.588, 'start': 0.12,...","[182.189, 152.634, 165.402, 135.936, 120.261, ...","[0.11, 0.012, 0.01, 0.012, 0.008, 0.012, 0.01,..."
3,"Hi, saya adalah pembantu AI anda, selamat berk...",open quote Yang itu kurungan terbuka denda R M...,open quote Yang itu kurungan terbuka denda R M...,None,prepare-dataset-normalizer-text-malay-news-ida...,idayu,0.776151,11.877007,2166069,"[{'end': 0.4, 'score': -1.74, 'start': 0.14, '...","[339.107, 230.672, 210.806, 278.877, 266.024, ...","[0.025, 0.008, 0.02, 0.04, 0.008, 0.014, 0.028..."
4,"Hi, saya adalah pembantu AI anda, selamat berk...","Cik, nak tanya, berapa lebar pintu masuk Cik? ...","Cik, nak tanya, berapa lebar pintu masuk Cik? ...",shafiqah-idayu-enhanced-v2-v2-trim.mp3,response-idayu-v2/160707.mp3,idayu,0.760246,5.212880,66815,"[{'end': 0.34, 'score': -0.103, 'start': 0.18,...","[222.045, 234.713, 231.469, 224.546, 214.563, ...","[0.015, 0.013, 0.033, 0.013, 0.008, 0.012, 0.0..."


In [7]:
# from neucodec import NeuCodec
 
# model = NeuCodec.from_pretrained("neuphonic/neucodec")
# _ = model.eval()

In [8]:
df['index'] = df.index.tolist()

In [9]:
rows = df[['normalized_generate_text', 'alignment', 'speaker', 'index']].to_dict(orient = 'records')

In [12]:
from tqdm import tqdm

def loop(rows):
    rows, _ = rows
    data = []
    for r in tqdm(rows):
        scores = []
        alignment = r['alignment']
        for no, a in enumerate(alignment):
            if len(a['text']) > 1 and a['score'] < -20:
                break
            if no > 0 and (a['start'] - alignment[no - 1]['end']) > 0.5:
                break
            if (a['end'] - a['start']) > 1.0:
                break
            scores.append(a['score'])
        if len(scores) != len(alignment):
            continue

        data.append(r)
    return data

In [13]:
filtered = loop((rows, 0))

100%|██████████| 1645455/1645455 [00:06<00:00, 236594.90it/s]


In [14]:
len(filtered)

659690

In [15]:
filtered_filtered = []
for r in filtered:
    r.pop('alignment')
    filtered_filtered.append(r)

In [16]:
# with open('neucodec/4.json') as fopen:
#     d = json.load(fopen)

In [19]:
import json

with open('prepared-Malaysian-TTS-v2.json', 'w') as fopen:
    json.dump(filtered_filtered, fopen)